## IMPORT LIBRARY AND PATH CONFIGURATION

In [ ]:
import sys
from pathlib import Path
import numpy as np

# Add python/ to PYTHONPATH (project root for Python code)
PROJECT_ROOT = Path(".").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Python project root:", PROJECT_ROOT)


## IMPORT MODULES

In [ ]:
from scripts.data_extraction_from_motive import Read_dataRB
from scripts.VirtualPoints_coordinate_extraction import export_virtual_points
from scripts.MotiveWorld_coordinate import Global_points
from scripts.UnityWorld_coordinate import Mesh_to_World_Pipeline
from scripts.Pipeline_Validation  import load_mesh_to_world, knee_flexion_angle, RMS_error 

### **READ RAW DATA FROM MOTIVE**

In [ ]:
bodies = {
    "Tibia": "tibiaTransform.csv",
    "Femur": "femurTransform.csv",
     "Patella": "PatellaTransform.csv",
}

Read_dataRB(
    src_csv="../data/csv_files/dataRigidBody.csv",
    output_dir="../data/csv_files",
    bodies=bodies
)


### **LOAD VP COORDIANTE ACQUIRED IN MATLAB**

In [ ]:
virtual_points = {
    "Femur": {
        "points": ["femurAFlocal", "femurLFlocal", "femurMFlocal"],
        "output": "femurVP.csv",
    },
    "Tibia": {
        "points": ["tibiaALTlocal", "tibiaAMTlocal", "tibiaLTlocal", "tibiaMTlocal"],
        "output": "tibiaVP.csv",
    },
    "Patella": {
        "points": ["patellaDPlocal", "patellaMPlocal", "patellaLPlocal", "patellaPPlocal"],
        "output": "patellaVP.csv"
    }
    }

export_virtual_points(mat_file="../data/matlab/virtualPoint_DEMO.mat",
                      output_dir="../data/csv_files",
                      virtual_points=virtual_points )

### **COMPUTATION OF MOTIVE GLOBAL COORDINATES USING q(w,x,y,z) & T(x,y,z) PROVIDED BY MOTIVE**

In [ ]:
Global_points(tibia_transform_csv="../data/csv_files/tibiaTransform.csv",
              femur_transform_csv="../data/csv_files/femurTransform.csv",
              patella_transform_csv="../data/csv_files/patellaTransform.csv",
              femur_vp_csv="../data/csv_files/femurVP.csv",
              tibia_vp_csv="../data/csv_files/tibiaVP.csv",
              patella_vp_csv="../data/csv_files/patellaVP.csv",
              output_dir="../data/csv_files",
              output_RT_dir="../data/Rigid_Transformation")

### **COMPUTATION OF R,T FROM MESH LOCAL REFERENCE FRAME TO MOTIVE GLOBAL WORLD AND COMPUTATION OF UNITY CSV FOR ANIMATION**

In [ ]:
Mesh_to_World_Pipeline(
              tibia_vp_csv="../data/csv_files/tibiaVP.csv",
              femur_vp_csv="../data/csv_files/femurVP.csv",
              patella_vp_csv="../data/csv_files/patellaVP.csv",
              tibia_transform_csv="../data/csv_files/tibiaTransform.csv",
              femur_transform_csv="../data/csv_files/femurTransform.csv",
              patella_transform_csv="../data/csv_files/patellaTransform.csv",
              output_dir="../data/csv_files",
              output_RT_dir="../data/Rigid_Transformation")

### **METRICS FOR VALIDARION AND GRAPHICS**

In [ ]:
# MESH LOCAL POINTS (BLENDER → MOTIVE)

Rot_B2M = np.diag([-1.0, 1.0, -1.0])

meshtibia_local = Rot_B2M @ np.array([
    [-0.02177037,  -0.2164885,   0.01993167],
    [ 0.03097912,   0.1695145,  -0.02298864],
    [-0.03774497,   0.1665102,   0.005437955],
], dtype=float).T

meshfemur_local = Rot_B2M @ np.array([
    [-0.01759300,   0.2254449,   0.02862103],
    [ 0.02254823,  -0.2074343,  -0.03284946],
    [-0.05463326,  -0.2044385,  -0.00376609],
], dtype=float).T

meshpatella_local = Rot_B2M @ np.array([
    [ 0.00104400,  -0.02765099,  0.005901016],
    [-0.01865400,  -0.00665200, -0.008743007],
    [ 0.01838301,  -0.01326200,  0.01198900],
], dtype=float).T

# LOAD MESH → WORLD TRANSFORMS (MOTIVE WORLD)

R_tibia,   T_tibia   = load_mesh_to_world("../data/Rigid_Transformation/mesh_to_world_tibia.npz")
R_femur,   T_femur   = load_mesh_to_world("../data/Rigid_Transformation/mesh_to_world_femur.npz")
R_patella, T_patella = load_mesh_to_world("../data/Rigid_Transformation/mesh_to_world_patella.npz")

# FLEXION ANGLE
angle = knee_flexion_angle(R_tibia, T_tibia, R_femur, T_femur, plot=True)
print(f"Starting angle: {angle.max():.0f}°")
print(f"Ending angle:   {angle.min():.0f}°")
print(f"Angle range: {angle.max() - angle.min():.0f}°\n")

# RMS ERROR
err_tibia, err_femur, err_patella = RMS_error(
    R_tibia, T_tibia,
    R_femur, T_femur,
    R_patella, T_patella,
    meshtibia_local,
    meshfemur_local,
    meshpatella_local,
    "../data/csv_files/tibiaGlobalPoints.csv",
    "../data/csv_files/femurGlobalPoints.csv",
    "../data/csv_files/patellaGlobalPoints.csv",
)



print(f"Tibia RMS  (mm): {err_tibia.mean() * 1000:.2f}")
print(f"Femur RMS  (mm): {err_femur.mean() * 1000:.2f}")
print(f"Patella RMS(mm): {err_patella.mean() * 1000:.2f}")
